In [1]:
import pandas as pd
import numpy as np

## Load and inspect raw data

Read a sample of each file first to check structure and confirm both halves share the same schema.

In [2]:
# Read first 1000 rows of each file just for inspection
web_1_sample = pd.read_csv("../data/raw/df_final_web_data_pt_1.csv", nrows=1000)
web_2_sample = pd.read_csv("../data/raw/df_final_web_data_pt_2.csv", nrows=1000)

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/df_final_web_data_pt_1.csv'

In [ ]:
web_1_sample.head()

In [ ]:
web_2_sample.head()

In [ ]:
web_1_sample.shape, web_2_sample.shape

In [ ]:
web_1_sample.info()

In [ ]:
web_2_sample.info()

In [ ]:
web_1_sample.columns

In [ ]:
web_2_sample.columns

Confirm that both files share the same columns before concatenating.

In [ ]:
web_1_sample.columns.equals(web_2_sample.columns)

##  Load full data and concatenate

In [3]:
web_1 = pd.read_csv("df_final_web_data_pt_1.csv")
web_2 = pd.read_csv("df_final_web_data_pt_2.csv")

df_web = pd.concat([web_1, web_2], ignore_index=True)

## Quality checks

In [4]:
df_web.shape

(755405, 5)

In [5]:
df_web.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04


In [ ]:
df_web.info()

In [ ]:
df_web["client_id"].nunique()

In [6]:
df_web.groupby("client_id").size().describe()

count    120157.000000
mean          6.286816
std           3.986973
min           1.000000
25%           5.000000
50%           5.000000
75%           7.000000
max         111.000000
dtype: float64

In [ ]:
df_web["process_step"].value_counts()

In [7]:
df_web.duplicated().sum()

np.int64(10764)

In [8]:
# Convert to datetime so we can compute durations and sort chronologically
df_web["date_time"] = pd.to_datetime(df_web["date_time"])
print(df_web["date_time"].dtype)

datetime64[us]


In [ ]:
df_web["process_step"].value_counts()

In [ ]:
df_web["process_step"].unique()

In [ ]:
df_web["date_time"].isna().sum()

## Sort

Sort by client → visit → time so that subsequent `shift()` operations correctly reference the previous event *within the same visit*.

In [9]:
df_web = df_web.sort_values(["client_id", "visit_id", "date_time"])

## Feature engineering

Build the columns needed for downstream analysis:
- `step_num` — numeric position of each step in the funnel
- `previous_step_num` — the step that came right before in the same visit
- `step_direction` — whether the user moved forward, backward, repeated, or just started
- `time_on_each_step` — seconds elapsed since the previous event in the same visit

In [10]:
# Map step names to numeric positions so we can compare them arithmetically
df_web["step_num"] = df_web["process_step"].map({"start": 0, "step_1": 1, "step_2": 2, "step_3": 3, "confirm": 4})

In [11]:
# For each event, get the step number of the previous event within the same visit
df_web['previous_step_num'] = df_web.groupby('visit_id')['step_num'].shift(1)

In [12]:
# NaN previous_step_num = first event of a visit
df_web["previous_step_num"].isna().sum()

np.int64(158095)

In [13]:
df_web["visit_id"].nunique()

158095

In [14]:
# Classify each event by direction relative to the previous event
# "first_visit" applies when there's no previous step (i.e., NaN)
conditions = [
    df_web["previous_step_num"] > df_web["step_num"],
    df_web["previous_step_num"] < df_web["step_num"],
    df_web["previous_step_num"] == df_web["step_num"]
]

labels = ["backward", "forward", "repeat"]

df_web["step_direction"] = np.select(conditions, labels, default="first_visit")

In [15]:
# Seconds between this event and the previous one in the same visit
df_web["time_on_each_step"] = (df_web["date_time"] - df_web.groupby("visit_id")["date_time"].shift(1)).dt.total_seconds()

In [28]:
time_per_step = df_web[df_web["process_step"] != "confirm"].groupby('process_step')['time_on_each_step'].mean().reset_index()

In [ ]:
## Save
df_web.to_csv("events_clean.csv", index=False)

In [29]:
time_per_step.sort_values(by="time_on_each_step", ascending=False)

,process_step,time_on_each_step
0,start,139.024150
3,step_3,99.382272
2,step_2,45.788994
1,step_1,39.439692


In [25]:
df_web

,client_id,visitor_id,visit_id,process_step,date_time,step_num,previous_step_num,step_direction,time_on_each_step
285515,169,201385055_71273495308,749567106_99161211863_557568,start,2017-04-12 20:19:36,0,NaN,first_visit,NaN
285514,169,201385055_71273495308,749567106_99161211863_557568,step_1,2017-04-12 20:19:45,1,0.0,forward,9.0
285513,169,201385055_71273495308,749567106_99161211863_557568,step_2,2017-04-12 20:20:31,2,1.0,forward,46.0
285512,169,201385055_71273495308,749567106_99161211863_557568,step_3,2017-04-12 20:22:05,3,2.0,forward,94.0
285511,169,201385055_71273495308,749567106_99161211863_557568,confirm,2017-04-12 20:23:09,4,3.0,forward,64.0
...,...,...,...,...,...,...,...,...,...
648533,9999875,738878760_1556639849,931268933_219402947_599432,step_1,2017-06-01 22:40:08,1,0.0,forward,7.0
648532,9999875,738878760_1556639849,931268933_219402947_599432,step_1,2017-06-01 22:41:28,1,1.0,repeat,80.0
648531,9999875,738878760_1556639849,931268933_219402947_599432,step_2,2017-06-01 22:41:47,2,1.0,forward,19.0
648530,9999875,738878760_1556639849,931268933_219402947_599432,step_3,2017-06-01 22:44:58,3,2.0,forward,191.0


## Initial Insights from Web Data

### 1. Sessions per Client
- Clients have multiple sessions on average.
- There is noticeable variability: some clients interact only once, while others return multiple times.
- This suggests different levels of engagement across users.

### 2. Funnel / Drop-off Behavior
- There is a clear drop-off at each step of the process:
  - Start → Step 1 → Step 2 → Step 3 → Confirm
- A significant portion of users does not reach the final confirmation step.
- This indicates potential friction points in the process that may need further investigation.

### 3. User Journey Patterns
- Most sessions follow the expected sequence (start → step progression → confirm).
- However, some irregular patterns exist (e.g. repeated steps or incomplete journeys).
- This suggests that not all users follow a clean linear path.

### 4. Data Structure Insight
- Raw web data is event-level (multiple rows per client).
- After aggregation, we successfully transformed it into a client-level dataset.
- Each client now has a clear behavioral profile (whether they reached each step).

---

## Note
These insights are based only on web interaction data.
They should be validated and extended after merging with experiment and client datasets.